## PREPROCESSING

### Objective

Perform the **ETL** process for the visualization project.

The goal is to provide relevant information and raise awareness in the community, especially for people who own motor vehicles and live in, or need to park their vehicles in, the city of Toronto. The visualizations will help users stay informed so they can better protect their vehicles.

To support this, a density map of incidents across **Toronto** will be displayed by neighbourhood. The **top 5 neighbourhoods** with the highest number of incidents will also be highlighted. In addition, a heatmap of incidents by day of week and hour of day will be created to provide more detailed awareness.

To do this, we need to process data from:

- Incident data from **Toronto Open Data**: https://open.toronto.ca/dataset/theft-from-motor-vehicle/

- Geospatial data for the map from **GeoJSON**: https://github.com/jasonicarter/toronto-geojson/raw/master/toronto_crs84.geojson



In [1]:
# import necessary libraries
import pandas as pd
from pathlib import Path
import geopandas as gpd
import requests
import os

In [2]:
# ignore any warnings
import warnings
warnings.filterwarnings("ignore")

First, download `toronto_crs84.geojson` and store it in `data/raw`

In [3]:
geojson_path = Path("../data/raw/toronto_neighbourhoods.geojson")
geojson_path.parent.mkdir(parents=True, exist_ok=True)

url = "https://github.com/jasonicarter/toronto-geojson/raw/master/toronto_crs84.geojson"
# load and write geojson file from the given url into /data/raw
response = requests.get(url)
with open(geojson_path, 'w') as f:
    f.write(response.text)

neighbourhoods = gpd.read_file(geojson_path)

print(neighbourhoods.columns)

Index(['AREA_S_CD', 'AREA_NAME', 'geometry'], dtype='object')


In [4]:
neighbourhoods.head()

,AREA_S_CD,AREA_NAME,geometry
0,097,Yonge-St.Clair (97),"POLYGON ((-79.39119 43.68108, -79.39141 43.680..."
1,027,York University Heights (27),"POLYGON ((-79.50529 43.75987, -79.50488 43.759..."
2,038,Lansing-Westgate (38),"POLYGON ((-79.43998 43.76156, -79.44004 43.761..."
3,031,Yorkdale-Glen Park (31),"POLYGON ((-79.43969 43.70561, -79.44011 43.705..."
4,016,Stonegate-Queensway (16),"POLYGON ((-79.49262 43.64744, -79.49277 43.647..."


Then, download `Thief From Motor Vehicle` data, store it and explore

In [5]:
# URL for Toronto Theft from Motor Vehicle CSV
url = "https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/1fc65d1e-7dae-4766-98dd-3b172e40a089/resource/a8b3ec01-2c82-41b8-81f3-eec5758e519f/download/theft-from-motor-vehicle%20-%204326.csv"

# Load directly to df
df = pd.read_csv(url)
print(f"Downloaded {df.shape[0]:,} rows, {df.shape[1]} columns")
print(df.head())

# Save to raw folder
output_path = Path("../data/raw/toronto_theft_mv.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Downloaded 104,965 rows, 27 columns
   _id EVENT_UNIQUE_ID REPORT_DATE    OCC_DATE  REPORT_YEAR REPORT_MONTH  \
0    1  GO-20141261501  2014-01-01  2014-01-01         2014      January   
1    2  GO-20141260616  2014-01-01  2014-01-01         2014      January   
2    3  GO-20141263677  2014-01-01  2013-12-31         2014      January   
3    4  GO-20149000010  2014-01-01  2013-12-31         2014      January   
4    5  GO-20141262668  2014-01-01  2013-12-31         2014      January   

   REPORT_DAY  REPORT_DOY  REPORT_DOW  REPORT_HOUR  ...  \
0           1           1  Wednesday             8  ...   
1           1           1  Wednesday             2  ...   
2           1           1  Wednesday            18  ...   
3           1           1  Wednesday             0  ...   
4           1           1  Wednesday            14  ...   

                                       LOCATION_TYPE PREMISES_TYPE  UCR_CODE  \
0  Single Home, House (Attach Garage, Cottage, Mo...         House      

The intention is to display the most recent data, so incidents from early 2025 to the present are most relevant for our visualization and awareness goals. Therefore, the dataset will be trimmed to focus on this period.

In [6]:
df_recent = df[df["OCC_YEAR"].isin([2025, 2026])]


In [7]:
# Count number of missing data
n_missing = df_recent.isna().any(axis=1).sum()
n_missing

np.int64(73)

In [8]:
len(df_recent)

6693

In [9]:
# Proportion of rows with any missing values
missing_proportion = n_missing * 100 / len(df_recent)
missing_proportion

np.float64(1.0906917675183028)

There are 73 observations with missing values across any column out of 6693 total 2025 data points (~1.1%). Therefore, removing these missing data points is acceptable for this visualization.

In [10]:
df_recent = df_recent.dropna(axis=0)

In [11]:
# double check na after dropping
df_recent.isna().sum()

_id                0
EVENT_UNIQUE_ID    0
REPORT_DATE        0
OCC_DATE           0
REPORT_YEAR        0
REPORT_MONTH       0
REPORT_DAY         0
REPORT_DOY         0
REPORT_DOW         0
REPORT_HOUR        0
OCC_YEAR           0
OCC_MONTH          0
OCC_DAY            0
OCC_DOY            0
OCC_DOW            0
OCC_HOUR           0
DIVISION           0
LOCATION_TYPE      0
PREMISES_TYPE      0
UCR_CODE           0
UCR_EXT            0
OFFENCE            0
CSI_CATEGORY       0
HOOD_158           0
LONG_WGS84         0
LAT_WGS84          0
geometry           0
dtype: int64

In [12]:
df_recent.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6620 entries, 98092 to 104964
Data columns (total 27 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   _id              6620 non-null   int64  
 1   EVENT_UNIQUE_ID  6620 non-null   object 
 2   REPORT_DATE      6620 non-null   object 
 3   OCC_DATE         6620 non-null   object 
 4   REPORT_YEAR      6620 non-null   int64  
 5   REPORT_MONTH     6620 non-null   object 
 6   REPORT_DAY       6620 non-null   int64  
 7   REPORT_DOY       6620 non-null   int64  
 8   REPORT_DOW       6620 non-null   object 
 9   REPORT_HOUR      6620 non-null   int64  
 10  OCC_YEAR         6620 non-null   float64
 11  OCC_MONTH        6620 non-null   object 
 12  OCC_DAY          6620 non-null   float64
 13  OCC_DOY          6620 non-null   float64
 14  OCC_DOW          6620 non-null   object 
 15  OCC_HOUR         6620 non-null   int64  
 16  DIVISION         6620 non-null   object 
 17  LOCATION_TYPE

For this visualization, the following columns are relevant:
"EVENT_UNIQUE_ID", "OCC_MONTH", "OCC_DAY", "OCC_DOW", "OCC_HOUR", "LONG_WGS84", "LAT_WGS84"

In [13]:
relevant_col = ["EVENT_UNIQUE_ID", "OCC_MONTH", "OCC_DAY", "OCC_DOW", "OCC_HOUR", "LONG_WGS84", "LAT_WGS84"]
relevant_data = df_recent[relevant_col].reset_index(drop=True)
relevant_data.columns = [col.lower() for col in relevant_col]
relevant_data.head()

,event_unique_id,occ_month,occ_day,occ_dow,occ_hour,long_wgs84,lat_wgs84
0,GO-20254813,January,1.0,Wednesday,13,-79.434285,43.657123
1,GO-20254618,January,1.0,Wednesday,16,-79.291282,43.693142
2,GO-20253514,January,1.0,Wednesday,0,-79.587090,43.753617
3,GO-20259000200,January,2.0,Thursday,18,-79.391265,43.671129
4,GO-20259890,January,1.0,Wednesday,21,-79.380493,43.664644


In [14]:
column_mapping = {"long_wgs84": "long", "lat_wgs84": "lat"}
relevant_data = relevant_data.rename(columns=column_mapping)
relevant_data.head()

,event_unique_id,occ_month,occ_day,occ_dow,occ_hour,long,lat
0,GO-20254813,January,1.0,Wednesday,13,-79.434285,43.657123
1,GO-20254618,January,1.0,Wednesday,16,-79.291282,43.693142
2,GO-20253514,January,1.0,Wednesday,0,-79.587090,43.753617
3,GO-20259000200,January,2.0,Thursday,18,-79.391265,43.671129
4,GO-20259890,January,1.0,Wednesday,21,-79.380493,43.664644


In [15]:
# Create preprocessed folder if it doesn't exist
# then save preprocessed data
os.makedirs("../data/preprocessed", exist_ok=True)
relevant_data.to_csv("../data/preprocessed/toronto_theft_mv_preprocessed.csv", index=False)
